# DeepEval Hallucination Evaluation

The purpose of this notebook is to run test cases of hallucination evaluations on datasets.

## Setup

In [ ]:
pip install --upgrade deepeval

In [69]:
from dotenv import load_dotenv
import os
import pandas as pd
import numpy as np
import anthropic
import sys
import io
import copy
from deepeval.models.base_model import DeepEvalBaseLLM
from deepeval.metrics import HallucinationMetric
from deepeval.test_case import LLMTestCase
from deepeval import evaluate
from sklearn.metrics import accuracy_score, precision_score, recall_score

In [70]:
# Load environment files and store API keys
load_dotenv()
anthropic_key = os.getenv("ANTHROPIC_KEY")
gemini_key = os.getenv("GEMINI_KEY")
openai_key = os.getenv("OPENAI_KEY")

## Data

In [71]:
# Read in dialogue data
file_path = "../../data/Hallucination/dialogue_data.json"
dialogue_data = pd.read_json(file_path, lines=True)

file_path = "../../data/Hallucination/general_data.json"
general_data = pd.read_json(file_path, lines=True)

file_path = "../../data/Hallucination/qa_data.json"
qa_data = pd.read_json(file_path, lines=True)

In [72]:
# Generate a random choice (True for right_answer, False for hallucinated_answer)
choice = np.random.rand(len(qa_data)) < 0.5

# Assign the chosen answer
qa_data["selected_answer"] = np.where(choice, qa_data["right_answer"], qa_data["hallucinated_answer"])

# Add a column indicating the source of the answer
qa_data["hallucinated_flag"] = np.where(choice, 0, 1)

## LLM Connection

In [101]:
class Claude(DeepEvalBaseLLM):
    """Class to implement Claude model for DeepEval"""
    def __init__(self, model, api_key):
        self.model = model
        self.api_key = api_key
        self.client = anthropic.Client(api_key=self.api_key)

    def load_model(self):
        return self.model

    def generate(self, prompt, max_tokens = 8192):
        response = self.client.messages.create(
            model=self.model,
            max_tokens=max_tokens,
            messages=[
                {"role": "user", "content": prompt}
            ]
        )
        return response.content[0].text

    async def a_generate(self, prompt, max_tokens = 8192):
        
        model = self.load_model()
        
        response = self.client.messages.create(
            model=self.model,
            max_tokens=max_tokens,
            messages=[
                {"role": "user", "content": prompt}
            ]
        )
        return response.content[0].text

    def get_model_name(self):
        return "Claude Model"

In [102]:
# Initialize the Claude model with appropriate settings
claude_model = "claude-3-5-sonnet-20240620"

# Create Claude model
claude_instance = Claude(model=claude_model, api_key=anthropic_key)

# Test functionality
print(claude_instance.generate("Hello, Claude"))

In [118]:
qa_data_test = copy.deepcopy(qa_data.head(100))

In [120]:
# Function to query the LLM with hallucination-aware prompt
def evaluate_hallucination(context, question, response):
   
   # prompt = f"""
   # You are an AI assistant that strictly answers questions based on the given context.
   # If the answer is not in the context, say "I don’t know based on the provided information."
   # Do NOT make up facts, infer missing details, or add extra information beyond the context.

   # Context:
   # {context}

   # Question:
   # {question}
   
   # Answer:
   # """

   # Generate response to the prompt
   # claude_instance.generate(prompt)
   
   # Create DeepEval test case
   test_case = LLMTestCase(
       input=f"\n\nHuman:{question}\n\nAssistant:",
       actual_output=f"\n\n{response}\n\nAssistant:",
       context=[f"\n\n{context}\n\nAssistant:"]
   )

   # Create hallucination evaluation metric
   metric = HallucinationMetric(model = claude_instance, threshold=0.5, include_reason=False)

   # Avoid unnecessary printing from DeepEval functions
   original_stdout = sys.stdout
   sys.stdout = io.StringIO()
   
   # Run evaluation using test case and metric
   evaluation = evaluate([test_case], [metric])

   # Avoid unnecessary printing from DeepEval functions
   sys.stdout = original_stdout
   
   # Store hallucination score
   hallucination_score = evaluation.test_results[0].metrics_data[0].score
   
   # Store reasoning for score
   reasoning = evaluation.test_results[0].metrics_data[0].reason
   
   # Return outputs
   return hallucination_score, reasoning
   # return response, hallucination_score, reasoning

In [ ]:
## Initialize list to hold results
#results = []
#
## Iterate through entire dataset
#for index, row in qa_data_test.iterrows():
#    # Get results
#    score, reasoning = evaluate_hallucination(row.knowledge, row.question, row.selected_answer)
#    # response, score, reasoning = evaluate_hallucination(row.knowledge, row.question, row.response)
#    
#    # Store results
#    results.append({
#        "question": row.question,
#        "context": row.knowledge,
#        "answer": row.selected_answer, # response,
#        "hallucination_score": score,
#        "reasoning": reasoning,
#        "hallucination_flag": row.hallucinated_flag
#    })

In [121]:
test_cases = []

for index, row in qa_data_test.iterrows():
    
    test_case = LLMTestCase(
        input=f"\n\nHuman:{row.question}\n\nAssistant:",
        actual_output=f"\n\n{row.selected_answer}\n\nAssistant:",
        context=[f"\n\n{row.knowledge}\n\nAssistant:"]
    )
    
    test_cases.append(test_case)
    
hallucination_metric = HallucinationMetric(model = claude_instance, threshold=0.5, include_reason=False)
    
# Avoid unnecessary printing from DeepEval functions
original_stdout = sys.stdout
sys.stdout = io.StringIO()

# Run evaluation using test case and metric
evaluation = evaluate(test_cases=test_cases, metrics=[hallucination_metric])

# Avoid unnecessary printing from DeepEval functions
sys.stdout = original_stdout

✨ You're running DeepEval's latest Hallucination Metric! (using Claude Model, strict=False, async_mode=True)...

Evaluating 100 test case(s) in parallel: |█████████▌| 96% (96/100) [Time Taken: 03:43,  2.33s/test case]


ValueError: Evaluation LLM outputted an invalid JSON. Please use a better evaluation model.

In [122]:
qa_results = pd.DataFrame({
    'knowledge': qa_data_test.knowledge,
    'question': qa_data_test.question,
    'selected_answer': qa_data_test.selected_answer,
    'hallucinated_flag': qa_data_test.hallucinated_flag,
    'hallucination_score': [i.metrics_data[0].score for i in evaluation.test_results]
})

ValueError: array length 20 does not match index length 100

In [ ]:
# evaluation_df.to_csv('../../results/Hallucination/DeepEval/qa_results.csv')

In [112]:
# qa_results = pd.read_csv('../../results/Hallucination/DeepEval/qa_results.csv')
qa_results['hallucination_pred'] = np.where(qa_results['hallucination_score'] > 0.5, 1, 0)

In [114]:
# Compute metrics
accuracy = accuracy_score(qa_results["hallucinated_flag"], qa_results["hallucination_pred"])
precision = precision_score(qa_results["hallucinated_flag"], qa_results["hallucination_pred"])
recall = recall_score(qa_results["hallucinated_flag"], qa_results["hallucination_pred"])

# Print results
print(f"Accuracy: {accuracy*100:.2f}%")
print(f"Precision: {precision*100:.2f}%")
print(f"Recall: {recall*100:.2f}%")

In [115]:
f"Accuracy: {accuracy*100:.2f}%"

'Accuracy: 85.00%'

In [116]:
f"Precision: {precision*100:.2f}%"

'Precision: 100.00%'

In [117]:
f"Recall: {recall*100:.2f}%"

'Recall: 75.00%'